# Pooling Gradients

AI & ML Foundations: Convolutional networks · *Mathematics for AI and Machine Learning* by Fuheng Wu

Max pooling sends the gradient only to each window's maximum; average pooling spreads it evenly (Chapter 11).

Companion code for the [interactive demo](https://math4ai.org/companion/ml/pooling-gradients/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch
import torch.nn.functional as F

# Twelve inputs, four windows of three (as in the demo)
x = torch.tensor([2.5, 4.0, 1.0, 3.0, 0.5, 3.5, 1.5, 1.5, 4.5, 2.0, 0.0, 1.0], requires_grad=True)

# ── Max pooling: the gradient goes only to each window's maximum ────────────
y = F.max_pool1d(x[None, None], 3)[0, 0]
y.backward(torch.ones_like(y))                    # upstream gradient 1 for every pooled output
print("max pool outputs   :", y.tolist())
print("grad (max pooling) :", x.grad.tolist())
print("non-zero entries   :", int((x.grad != 0).sum()), "of", x.numel(), "-> a sparse Jacobian")

# ── Average pooling: every input in a window gets 1/k ───────────────────────
x.grad = None
y = F.avg_pool1d(x[None, None], 3)[0, 0]
y.backward(torch.ones_like(y))
print("\ngrad (avg pooling) :", [round(g, 4) for g in x.grad.tolist()])

# ── Ties: the book's convention splits the gradient between tied maxima ─────
xt = torch.tensor([1.0, 3.0, 3.0], requires_grad=True)
F.max_pool1d(xt[None, None], 3)[0, 0].sum().backward()
print("\ntie [1, 3, 3], PyTorch max_pool1d gradient:", xt.grad.tolist(), "(PyTorch picks one index)")
xt.grad = None
xt.max().backward()
print("tie [1, 3, 3], torch.max gradient        :", xt.grad.tolist(), "(torch.max splits evenly: 1/2 each)")

# ── The full Jacobian of max pooling ────────────────────────────────────────
J = torch.autograd.functional.jacobian(lambda v: F.max_pool1d(v[None, None], 3)[0, 0], x.detach())
print("\nJacobian (4 outputs x 12 inputs):")
print(J.int())

max pool outputs   : [4.0, 3.5, 4.5, 2.0]
grad (max pooling) : [0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0]
non-zero entries   : 4 of 12 -> a sparse Jacobian

grad (avg pooling) : [0.3333, 0.3333, 0.3333, 0.3333, 0.3333, 0.3333, 0.3333, 0.3333, 0.3333, 0.3333, 0.3333, 0.3333]

tie [1, 3, 3], PyTorch max_pool1d gradient: [0.0, 1.0, 0.0] (PyTorch picks one index)
tie [1, 3, 3], torch.max gradient        : [0.0, 0.5, 0.5] (torch.max splits evenly: 1/2 each)

Jacobian (4 outputs x 12 inputs):
tensor([[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]], dtype=torch.int32)
